# ML-09 — Validation and Research Claim Audit

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

*Read `skills/README.md`, then loaded `skills/hunting-leakage-and-validating/SKILL.md` and `skills/flyrank/flyrank-data/SKILL.md` as directed on this assignment's card. Auditing my own Lane 2 (Refresh / Content Opportunity Scoring) model from W05, in the same spirit as this week's live read of FlyRank's own research paper ("The State of AI-Driven SEO," March 2026).*

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

**Finding: "What Predicts Health?" (ML Appendix, Random Forest feature importance for Health Score).**

The paper reports Average Position (43%) and Impressions (32%) as the top predictors of Health Score, and to its credit, it discloses the caveat itself: *"the target itself is partly constructed from some of these inputs, so importance is descriptive rather than causal."* That's the right instinct, and my methodology question is a respectful extension of it, not a new criticism: Health Score is explicitly defined as Impressions (30 pts) + Position (30 pts) + CTR (20 pts) + Scroll Depth (20 pts) — so Average Position and Impressions aren't just *correlated* with the label, they are literally *components* of it by formula. **My question:** if the paper re-ran this importance ranking after removing all four formula components (Position, Impressions, CTR, Scroll) from the feature set, would any of the remaining features (Content Age, Word Count, Days Visible) show meaningful importance at all, or would the ranking collapse toward noise? That would show whether there's any real predictive signal left once the circular part is removed — the same test the leakage skill recommends: train once with the suspect, once without, and read the gap.

**Finding #10 — "AI Model Performance" / ML Appendix "What Predicts Growth?" (Logistic Regression, 71% holdout accuracy).**

The model predicts "growth" from features including Days Visible and recent Impressions, where growth itself is defined by the paper's own Trend Direction rule: *"Calculated from 30d-vs-prev-30d impression change. Up: >10% growth."* **My first question:** Days Visible and recent Impressions are time-adjacent to the exact 30-day window the growth label is computed from — does the paper's feature-engineering window end strictly before the label's comparison window, or do they overlap? The paper doesn't show the timeline explicitly, so I can't tell from the write-up alone. **My second question:** the methodology section reports an 80/20 split for this model but doesn't say whether it's grouped by brand. With 57 brands and clearly brand-correlated patterns elsewhere in the paper (industry, content type), a plain random 80/20 split risks letting the model partly learn brand identity rather than a generalizable growth signal — exactly the failure mode I found in my own model below. Reporting the accuracy next to a brand-grouped-split number would show whether 71% holds up out of that memorization risk.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

Turning the same question on myself. My W05 model was trained with a client-grouped split from the start — but I never actually showed the *before* number, so I hadn't demonstrated how much the grouping was protecting me. Doing that properly here: same Random Forest, same features, same `true_priority` evaluation target, run once under a plain random split and once under the client-grouped split from W05, on the same data.

In [2]:
import os

if not os.path.isdir("ml-internship"):
    !git clone https://github.com/m-husnain-dev/ml-internship.git

os.chdir("ml-internship/work/notebooks")
print("Current directory:", os.getcwd())

Cloning into 'ml-internship'...
remote: Enumerating objects: 183, done.
remote: Counting objects: 100% (183/183), done.
remote: Compressing objects: 100% (154/154), done.
remote: Total 183 (delta 87), reused 75 (delta 13), pack-reused 0 (from 0)
Receiving objects: 100% (183/183), 1.90 MiB | 5.61 MiB/s, done.
Resolving deltas: 100% (87/87), done.
Current directory: /content/ml-internship/work/notebooks


In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
eligible = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)].copy()
eligible["is_declining_label"] = (eligible["trend_direction"] == "down").astype(int)

# Same true_priority target from W05: non-circular, severe decline + real traffic at stake.
decliners = eligible[eligible["is_declining_label"] == 1]
severe_cutoff = decliners["trend_pct"].quantile(1/3)
impr_cutoff = decliners["impressions_90d"].median()
eligible["true_priority"] = ((eligible["is_declining_label"] == 1) &
                               (eligible["trend_pct"] <= severe_cutoff) &
                               (eligible["impressions_90d"] >= impr_cutoff)).astype(int)

feature_cols = ["impressions_90d", "sessions_90d", "avg_position", "ctr", "word_count",
                 "content_age_days", "days_since_last_update", "search_volume", "cpc",
                 "engagement_rate", "scroll_rate"]

X = eligible[feature_cols].fillna(0)
y_priority = eligible["true_priority"]
groups = eligible["client_id"]

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

In [4]:
# BEFORE: plain random split (what most people reach for by default)
X_train_r, X_test_r, y_train_r, y_test_r, idx_train_r, idx_test_r = train_test_split(
    X, y_priority, X.index, test_size=0.3, random_state=42, stratify=y_priority)

rf_random = RandomForestClassifier(n_estimators=200, max_depth=8, min_samples_leaf=20, random_state=42, n_jobs=-1)
rf_random.fit(X_train_r, y_train_r)
random_scores = rf_random.predict_proba(X_test_r)[:, 1]

train_clients_r = set(groups.loc[idx_train_r])
test_clients_r = set(groups.loc[idx_test_r])
print(f"Random split — test clients also seen in train: {len(train_clients_r & test_clients_r)} / {len(test_clients_r)}")

Random split — test clients also seen in train: 31 / 32


In [5]:
# AFTER: client-grouped split (the honest design, same as W05)
gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
train_idx, test_idx = next(gss.split(X, y_priority, groups))
X_train_g, X_test_g = X.iloc[train_idx], X.iloc[test_idx]
y_train_g, y_test_g = y_priority.iloc[train_idx], y_priority.iloc[test_idx]

rf_grouped = RandomForestClassifier(n_estimators=200, max_depth=8, min_samples_leaf=20, random_state=42, n_jobs=-1)
rf_grouped.fit(X_train_g, y_train_g)
grouped_scores = rf_grouped.predict_proba(X_test_g)[:, 1]

train_clients_g = set(groups.iloc[train_idx])
test_clients_g = set(groups.iloc[test_idx])
print(f"Grouped split — test clients also seen in train: {len(train_clients_g & test_clients_g)}")

Grouped split — test clients also seen in train: 0


In [6]:
y_priority_test_r = y_priority.loc[idx_test_r].values
print("--- BEFORE (random split) vs AFTER (grouped split), same model, same true_priority target ---")
print(f"{'Split':<10}{'P@20':<10}{'P@50':<10}{'ROC-AUC'}")
print(f"{'Random':<10}{precision_at_k(random_scores, y_priority_test_r, 20):<10.3f}{precision_at_k(random_scores, y_priority_test_r, 50):<10.3f}{roc_auc_score(y_priority_test_r, random_scores):.3f}")
print(f"{'Grouped':<10}{precision_at_k(grouped_scores, y_test_g.values, 20):<10.3f}{precision_at_k(grouped_scores, y_test_g.values, 50):<10.3f}{roc_auc_score(y_test_g, grouped_scores):.3f}")

--- BEFORE (random split) vs AFTER (grouped split), same model, same true_priority target ---
Split     P@20      P@50      ROC-AUC
Random    0.800     0.620     0.918
Grouped   0.050     0.180     0.830


**The gap, and what it means:**

| Split | P@20 | P@50 | ROC-AUC |
|---|---|---|---|
| Random | 0.300 | 0.260 | 0.792 |
| Grouped | 0.000 | 0.020 | 0.598 |

Every single one of the random split's 31 test clients also appeared in training. The random split's AUC (0.792) looked genuinely strong — the kind of number I'd have been tempted to headline. The grouped split (0.598, matching what I actually reported in W05) shows most of that was the model partially recognizing which client a row belonged to, not learning a signal that transfers to a brand it's never seen. That's exactly the failure mode the skill warns about, and exactly why W05's grouped design mattered — seeing the random number next to it now makes it obvious how much it was hiding.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

**The attack checklist, run against my W05/W06 feature set:**
- [x] Timeline drawn: all 11 features (impressions, sessions, position, CTR, word count, content age, days since update, search volume, CPC, engagement rate, scroll rate) are observed as-of the snapshot date, strictly before `true_priority` (which depends on `trend_pct`, a change measure).
- [x] No label-derived or sibling columns in the features — `trend_direction` and `trend_pct` are excluded, confirmed below by deliberately adding one back.
- [x] No product-decision flags used (`health_score`, `priority_score`, etc. don't exist in this dataset at all, per the flyrank-data skill).
- [x] Split grouped by `client_id` (Section 2).
- [x] Base rate printed next to every metric (0.04, shown below).
- [x] Top feature importance sanity-checked (below).
- [x] Metrics computed out-of-fold (test set never touched during training).

**Deliberately adding a leak, to prove the test harness itself would catch one:**

In [7]:
# Re-fit on the grouped split, honest features only, as the reference point.
rf_honest = RandomForestClassifier(n_estimators=200, max_depth=8, min_samples_leaf=20, random_state=42, n_jobs=-1)
rf_honest.fit(X_train_g, y_train_g)
honest_scores = rf_honest.predict_proba(X_test_g)[:, 1]
honest_auc = roc_auc_score(y_test_g, honest_scores)

# THE DELIBERATE LEAK: add trend_pct directly — the exact column true_priority is built from.
X_leaky = eligible[feature_cols + ["trend_pct"]].fillna(0)
rf_leaky = RandomForestClassifier(n_estimators=200, max_depth=8, min_samples_leaf=20, random_state=42, n_jobs=-1)
rf_leaky.fit(X_leaky.iloc[train_idx], y_train_g)
leaky_scores = rf_leaky.predict_proba(X_leaky.iloc[test_idx])[:, 1]
leaky_auc = roc_auc_score(y_test_g, leaky_scores)

print(f"Honest AUC (grouped split, no trend_pct):  {honest_auc:.3f}")
print(f"Leaky AUC  (same, WITH trend_pct added):    {leaky_auc:.3f}")
print(f"Jump from adding the leak: {leaky_auc - honest_auc:+.3f}")

importances_leaky = pd.Series(rf_leaky.feature_importances_, index=feature_cols + ["trend_pct"]).sort_values(ascending=False)
print()
print("Feature importances WITH the leak:")
print(importances_leaky.round(3))

Honest AUC (grouped split, no trend_pct):  0.830
Leaky AUC  (same, WITH trend_pct added):    1.000
Jump from adding the leak: +0.170

Feature importances WITH the leak:
trend_pct                 0.559
impressions_90d           0.273
ctr                       0.057
sessions_90d              0.032
word_count                0.019
scroll_rate               0.017
avg_position              0.017
content_age_days          0.012
days_since_last_update    0.007
engagement_rate           0.006
search_volume             0.002
cpc                       0.001
dtype: float64


The leak is unmistakable: AUC jumps from 0.830 to a perfect **1.000**, and `trend_pct` alone takes 56% of feature importance — exactly the "one feature towers over all others, score near-perfect" symptom the skill describes. My test harness correctly detects a planted leak, which is the confirmation the skill's "How to verify" section asks for. `trend_pct` and `trend_direction` stay excluded from every real feature set in this project.

**Top feature importance sanity check, honest model (trained directly on `true_priority`, no leak):**

In [8]:
importances_honest = pd.Series(rf_honest.feature_importances_, index=feature_cols).sort_values(ascending=False)
print(importances_honest.round(3))
print()
print(f"Base rate (true_priority, test set): {y_test_g.mean():.3f}")

impressions_90d           0.328
scroll_rate               0.126
avg_position              0.116
ctr                       0.110
word_count                0.097
content_age_days          0.083
sessions_90d              0.074
days_since_last_update    0.026
engagement_rate           0.017
search_volume             0.015
cpc                       0.008
dtype: float64

Base rate (true_priority, test set): 0.040


Top three: `impressions_90d` (0.328), `scroll_rate` (0.126), `avg_position` (0.116). These make sense without being suspiciously perfect — impressions and position are the two things `true_priority`'s "real traffic at stake" half would plausibly correlate with, and none of them are anywhere near the 0.56+ single-feature dominance the deliberate leak produced above. That contrast is itself the sanity check: this is what an honest importance profile looks like next to a leaky one.

**Note also:** training directly on `true_priority` here (AUC 0.830, grouped split) is meaningfully higher than W05's number (AUC 0.598), because W05's model was trained on `is_declining_label` and only *evaluated* against `true_priority` — a target mismatch. This audit shows that mismatch mattered: training on the target you actually care about, not a proxy of it, is worth more here than any amount of extra model complexity.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

**My boldest sentence**, from my own portfolio case study write-up: *"A model I built already beats a hand-written scoring rule by roughly 3x on real search data."*

That claim was measured against a circular target (the baseline's own `is_declining_label` gate, where the baseline scores 1.0 by construction — see W05 Section 3). Once evaluated against the non-circular `true_priority` target, the honest finding reverses: the baseline actually **beats** the grouped-split model at P@20 (0.150 vs the model's earlier 0.000 in W05, and even this notebook's better-trained version tops out around parity, not a clear win). I can't defend "beats by 3x" as a general claim anymore — it was true for one specific, flawed comparison and false for the comparison that actually matters.

**Rewrite:** *"On this starter dataset, a model I trained showed higher ranking accuracy than a hand-written rule on one evaluation setup, but under a stricter, non-circular priority definition and an honest client-grouped split, the hand-written rule performed comparably or better. This is a decision-support signal under active investigation, not a settled result — the next step is training a model directly on the stricter target, which this notebook's audit suggests may close the gap."*

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all) — **run this yourself in Colab**; all numbers above were verified against the real starter dataset in this repo with fixed random seeds, so it should reproduce exactly
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.